In [1]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.constr_methods import ConstrSolverMethod
from optim.constrained.ralm import RalmCfg
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem
from testing.testing_metrics import euclid_metric, scaled_metric, coupled_metric, asymmetric_metric

In [2]:
methods = [
    ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    # ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    # ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    # ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
    0.5, 1., 1.5
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 20

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])
cntr_center = np.array([-2., 3., 1.])
cntr_radius = 3.

# metrics to use in testing

metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    (asymmetric_metric, "asymmetric"),
]


In [3]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm
cntr_center /= max_starting_norm
cntr_radius /= max_starting_norm

In [4]:
# configure the constrained configurations
rgd_subsolver_cfg = RiemGradDescentCfg()
rtr_subsolver_cfg = RiemTrustRegionCfg()

ralm_cfg_subsolver_rgd = RalmCfg()
ralm_cfg_subsolver_rgd.subsolver_method = SubsolverMethod.RIEM_GRAD_DESCENT
ralm_cfg_subsolver_rgd.subsolver_cfg = rgd_subsolver_cfg

ralm_cfg_subsolver_rtr = RalmCfg()
ralm_cfg_subsolver_rtr.subsolver_method = SubsolverMethod.RIEM_TRUST_REGION
ralm_cfg_subsolver_rtr.subsolver_cfg = rtr_subsolver_cfg

optimizers = [
    ((ConstrSolverMethod.RALM, ralm_cfg_subsolver_rgd), "ralm_rgd"),
    ((ConstrSolverMethod.RALM, ralm_cfg_subsolver_rtr), "ralm_rtr")
]



In [5]:
# root directory to store output files inside
base_data_dir = Path("../data/constrained")

In [ ]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    trials_region_center = cntr_center * scaling
    trials_region_radius = cntr_radius * scaling

    # create the problem
    cost, region_ineqs = create_problem(torch.tensor(trials_cost_center), [[(torch.tensor(trials_region_center), trials_region_radius)]])
    region_ineq = region_ineqs[0]

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, [region_ineq], [], p0, compute_mfld, optimizer_cfg)
        # print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials:   0%|          | 0/20 [00:00<?, ?it/s]/home/samuel/Documents/School/UWaterloo_MASc/Courses/ECE_602/project/ece602-mfld-optim-taylor-approx/.venv/lib/python3.14/site-packages/torch/utils/_device.py:116: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)

Trials:   5%|▌         | 1/20 [00:41<13:06, 41.39s/it]

	Saving to ../data/constrained/ralm_rgd/metric_euclid__scaling_0.5__trial_0__geod_method_ivpbvp.pt



Trials:  10%|█         | 2/20 [01:23<12:31, 41.78s/it]

	Saving to ../data/constrained/ralm_rgd/metric_euclid__scaling_0.5__trial_1__geod_method_ivpbvp.pt



Trials:  15%|█▌        | 3/20 [02:04<11:44, 41.44s/it]

...

	Saving to ../data/constrained/ralm_rgd/metric_euclid__scaling_0.5__trial_6__geod_method_ivpbvp.pt



Trials:  40%|████      | 8/20 [05:29<08:13, 41.09s/it]

	Saving to ../data/constrained/ralm_rgd/metric_euclid__scaling_0.5__trial_7__geod_method_ivpbvp.pt



Trials:  45%|████▌     | 9/20 [06:10<07:32, 41.14s/it]

	Saving to ../data/constrained/ralm_rgd/metric_euclid__scaling_0.5__trial_8__geod_method_ivpbvp.pt
